<a href="https://colab.research.google.com/github/kdimbu-spec/Intelligent-Agents-Portfolio/blob/main/AcademicResearchAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The command installs all four libraries

### groq: Python client for interacting with the Groq AI API and running LLM-based applications.

### gradio: Creates simple web interfaces for AI models, demos, and applications.

### requests: Sends HTTP requests to APIs and websites to retrieve or send data.

### pandas: Data analysis and manipulation library using DataFrames.




In [1]:
!pip -q install groq gradio requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["GROQ_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [3]:
from groq import Groq

client = Groq(
    api_key=os.environ["GROQ_KEY"]
)

MODEL = "allam-2-7b"

In [4]:
print('Available Groq models:')
for model in client.models.list().data:
    print(model.id)

Available Groq models:
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
openai/gpt-oss-120b
allam-2-7b


In [5]:
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class ResearchState:

    research_question: str

    search_terms: List[str] = field(default_factory=list)

    raw_results: List[Dict[str, Any]] = field(default_factory=list)

    processed_results: List[Dict[str, Any]] = field(default_factory=list)

    report: str = ""

    retry_count: int = 0

    max_retries: int = 3

    sufficient: bool = False

    status: str = "INITIALISED"

    journal: List[str] = field(default_factory=list)

In [6]:
import json
import time
from groq import RateLimitError

def call_llm(prompt, temperature=0.2, max_retries=5):
    retries = 0
    delay = 1  # Initial delay in seconds

    while retries <= max_retries:
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an academic research planning agent."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temperature
            )
            return response.choices[0].message.content
        except RateLimitError as e:
            retries += 1
            print(f"Rate limit exceeded. Retrying in {delay} seconds... (Attempt {retries}/{max_retries})")
            time.sleep(delay)
            delay *= 2  # Exponential backoff
            if retries > max_retries:
                raise e # Re-raise if max retries are exceeded
        except Exception as e:
            # Handle other potential errors without retrying (or implement specific retry logic for them)
            raise e


In [7]:
def planning_agent(state):

    prompt = f"""
You are the Planning Agent in a multi-agent academic
research system.

Research question:
{state.research_question}

Generate 5 precise academic search queries that can be
used to retrieve relevant scholarly literature.

The queries should cover:
1. The main research topic
2. Alternative terminology
3. Relevant methodology
4. Related theoretical concepts
5. Recent research

Return ONLY valid JSON in this format:

{{
    "queries": [
        "query 1",
        "query 2",
        "query 3",
        "query 4",
        "query 5"
    ]
}}
"""

    response = call_llm(prompt)

    try:
        data = json.loads(response)
        state.search_terms = data["queries"]

    except Exception:
        state.search_terms = [
            state.research_question
        ]

    state.journal.append(
        f"Planning Agent generated {len(state.search_terms)} search queries."
    )

    state.status = "PLANNING_COMPLETE"

    return state

In [8]:
import requests

def search_openalex(query, per_page=5):

    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": per_page
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    results = []

    for item in data.get("results", []):

        authors = []

        for author in item.get("authorships", []):
            author_name = author.get("author", {}).get("display_name")

            if author_name:
                authors.append(author_name)

        results.append({
            "title": item.get("title"),
            "authors": authors,
            "year": item.get("publication_year"),
            "doi": item.get("doi"),
            "url": item.get("primary_location", {}).get("landing_page_url"),
            "source": "OpenAlex"
        })

    return results

In [9]:
def retrieval_agent(state):

    all_results = []

    for query in state.search_terms:

        try:

            results = search_openalex(
                query,
                per_page=5
            )

            all_results.extend(results)

            state.journal.append(
                f"Retrieved {len(results)} results for: {query}"
            )

        except Exception as e:

            state.journal.append(
                f"Retrieval error for '{query}': {str(e)}"
            )

    state.raw_results = all_results

    state.status = "RETRIEVAL_COMPLETE"

    return state

In [10]:
def deduplicate_results(results):

    unique = {}

    for item in results:

        title = item.get("title")

        if not title:
            continue

        key = title.lower().strip()

        if key not in unique:
            unique[key] = item

    return list(unique.values())

In [11]:
def processing_agent(state):

    state.raw_results = deduplicate_results(
        state.raw_results
    )

    state.journal.append(
        f"Deduplication completed. "
        f"{len(state.raw_results)} unique papers remain."
    )

    return state

In [12]:
def summarise_paper(paper):

    prompt = f"""
You are the Processing Agent in an academic research
system.

Analyse the following academic paper metadata.

Title:
{paper.get("title")}

Authors:
{", ".join(paper.get("authors", []))}

Year:
{paper.get("year")}

Provide:

1. Research relevance
2. Likely contribution
3. Methodology if identifiable
4. A short academic summary

Do not invent information that is not present.
"""

    return call_llm(prompt, temperature=0.1)

In [13]:
def process_papers(state, limit=10):

    processed = []

    for paper in state.raw_results[:limit]:

        summary = summarise_paper(paper)

        paper_copy = paper.copy()

        paper_copy["summary"] = summary

        processed.append(paper_copy)

    state.processed_results = processed

    state.journal.append(
        f"Processed {len(processed)} academic papers."
    )

    return state

In [14]:
def calculate_relevance(paper, question):

    title = paper.get("title", "").lower()
    question_words = set(
        question.lower().split()
    )

    title_words = set(title.split())

    overlap = question_words.intersection(title_words)

    score = len(overlap)

    return score

In [15]:
def rank_results(state):

    for paper in state.processed_results:

        paper["relevance_score"] = calculate_relevance(
            paper,
            state.research_question
        )

    state.processed_results.sort(
        key=lambda x: x["relevance_score"],
        reverse=True
    )

    state.journal.append(
        "Research papers ranked by relevance."
    )

    return state

In [16]:
MIN_RESULTS = 8

In [17]:
def check_sufficiency(state):

    number_of_results = len(
        state.processed_results
    )

    if number_of_results >= MIN_RESULTS:

        state.sufficient = True
        state.status = "RESULTS_SUFFICIENT"

    else:

        state.sufficient = False
        state.status = "RESULTS_INSUFFICIENT"

    state.journal.append(
        f"Sufficiency check: "
        f"{number_of_results} results."
    )

    return state

In [18]:
def can_retry(state):

    return state.retry_count < state.max_retries

In [19]:
def retry_planning(state):

    state.retry_count += 1

    state.journal.append(
        f"Retry {state.retry_count} initiated."
    )

    return planning_agent(state)

In [20]:
def run_research(question):

    state = ResearchState(
        research_question=question
    )

    planning_agent(state)

    while True:

        retrieval_agent(state)

        processing_agent(state)

        # Reduce the limit of processed papers to avoid exceeding context window
        process_papers(state, limit=5)

        rank_results(state)

        check_sufficiency(state)

        # ----------------------------
        # DECISION NODE
        # ----------------------------

        if state.sufficient:

            break

        if not can_retry(state):

            state.status = (
                "MAX_RETRIES_REACHED"
            )

            break

        retry_planning(state)

    storage_agent(state)

    return state

In [21]:
def storage_agent(state):

    papers_text = ""

    for i, paper in enumerate(
        state.processed_results,
        start=1
    ):

        papers_text += f"""

Paper {i}
Title: {paper.get("title")}
Authors: {", ".join(paper.get("authors", []))}
Year: {paper.get("year")}
Source: {paper.get("source")}
URL: {paper.get("url")}

Summary:
{paper.get("summary")}
"""

    prompt = f"""
You are the Storage and Report Agent.

Research question:
{state.research_question}

Using the research results below, produce a structured
academic research report.

Include:

# Research Question

# Search Strategy

# Key Literature

# Synthesis of Findings

# Research Gaps

# Conclusion

# References

Do not fabricate references or findings.

Research results:
{papers_text}
"""

    state.report = call_llm(
        prompt,
        temperature=0.1
    )

    state.status = "COMPLETED"

    state.journal.append(
        "Storage Agent generated final research report."
    )

    return state

In [22]:
import json

def save_journal(state):

    journal_data = {
        "research_question":
            state.research_question,

        "search_terms":
            state.search_terms,

        "retry_count":
            state.retry_count,

        "status":
            state.status,

        "journal":
            state.journal,

        "results":
            state.processed_results
    }

    with open(
        "research_journal.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            journal_data,
            f,
            indent=2,
            ensure_ascii=False
        )

In [23]:
question = """
How effective are machine learning techniques
in detecting COVID-19 misinformation on social
media platforms?
"""

state = run_research(question)

In [24]:
print(state.status)

COMPLETED


In [25]:
print("Retries:", state.retry_count)

Retries: 3


In [26]:
print("Papers:", len(state.processed_results))

Papers: 5


In [27]:
print(state.report)

Title: Machine Learning-Based Misinformation Detection in Social Media: A Case Study on COVID-19 Misinformation on Twitter
Authors: Mohammed Al-Ali, Ahmed Al-Ali, and Abdulrahman Al-Ali
Year: 2022
Source: OpenAlex
URL: https://doi.org/10.1007/s10501-022-0001-4


1. Research Relevance:
The COVID-19 pandemic has highlighted the importance of detecting misinformation on social media platforms, particularly Twitter. Misinformation can have severe consequences on public health and public opinion, making the topic relevant to the current global health crisis.

2. Likely Contribution:
The paper is likely to contribute to the understanding of misinformation detection techniques in social media platforms during pandemics, specifically on Twitter. By analyzing the effectiveness of machine learning-based methods, the study aims to provide insights into the challenges, trends, and gaps in the field, guiding future research directions.

3. Methodology if Identifiable:
The authors will likely employ

In [28]:
!pip -q install gradio

In [29]:
import gradio as gr

In [30]:
def research_interface(question):

    if not question.strip():

        return (
            "Please enter a research question.",
            "",
            ""
        )

    state = run_research(question)

    report = state.report

    journal = "\n".join(
        state.journal
    )

    status = f"""
Status: {state.status}

Retries: {state.retry_count}

Papers processed:
{len(state.processed_results)}
"""

    return report, journal, status

In [31]:
with gr.Blocks(
    title="Academic Research Multi-Agent System"
) as demo:

    gr.Markdown(
        """
        # 🎓 Academic Research Multi-Agent System

        An LLM-powered multi-agent system for
        academic literature discovery, processing
        and research report generation.
        """
    )

    question = gr.Textbox(
        label="Research Question",
        placeholder=(
            "Enter your academic research question..."
        ),
        lines=4
    )

    run_button = gr.Button(
        "🔎 Start Research"
    )

    status = gr.Textbox(
        label="System Status",
        lines=5
    )

    report = gr.Markdown(
        label="Research Report"
    )

    journal = gr.Textbox(
        label="Agent Activity Journal",
        lines=15
    )

    run_button.click(
        fn=research_interface,
        inputs=question,
        outputs=[
            report,
            journal,
            status
        ]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://647753bac887022850.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:
def save_report(state):

    with open(
        "final_research_report.md",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(state.report)

    return state